# Die Kontrollen der Strecke, an einem Fall durchgespielt

**Zweck.** Dieses Projekt prüft an vielen Stellen, ob die Daten halten, was sie
behaupten. Diese Prüfungen liegen dort, wo sie wirken: im Ingestionsmodul, in
der Ablage, in Silver, in Gold, in der Orchestrierung. Das ist richtig so, aber
es macht sie unübersichtlich. Wer wissen will, *was eigentlich kontrolliert
wird*, muss zehn Pakete durchsehen.

Hier stehen sie an einem Ort, in der Reihenfolge, in der sie greifen. Jede wird
**zweimal vorgeführt**: einmal mit Daten, die sie durchlässt, und einmal mit
Daten, die sie fangen soll. Die zweite Vorführung ist die wichtigere. Eine
Kontrolle, von der man nur gesehen hat, dass sie schweigt, hat man nicht
gesehen.

Am Schluss reist ein einzelner Fall durch alle Schichten, und es steht
daneben, welche Kontrolle ihn dabei angesehen hat und was sie gesagt hat.

**Ausgeführt wird echter Code.** Alle Funktionen hier stammen aus
`src/crime/`; nichts ist nachgebaut. Gebaute Daten braucht das Notebook nicht,
es erzeugt seine Beispiele selbst. Damit läuft es überall und jederzeit.

In [ ]:
import hashlib
import json
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config").exists():
    raise RuntimeError("Projektwurzel nicht gefunden")
sys.path.insert(0, str(PROJECT_ROOT / "src" / "crime"))

import duckdb

print(f"Projektwurzel: {PROJECT_ROOT}")
print(f"DuckDB {duckdb.__version__}")

## Zwei Arten von Kontrolle

Die Unterscheidung zieht sich durch das ganze Projekt, und sie ist eine
Entscheidung, keine Eigenschaft der Daten.

| | Was sie tut | Wann sie richtig ist |
| --- | --- | --- |
| **hart** | bricht ab, nichts wird geschrieben | wenn etwas **still** schieflaufen kann: ein doppelter Schlüssel, ein verlorener Fall, eine unvollständige Ladung |
| **weich** | schreibt das Ergebnis in eine Spalte und zählt | wenn etwas **unvollständig** ist: eine fehlende Koordinate, eine unsichere Zuordnung |

Der Satz dazu steht im Docstring von `silver_pipeline.build_crime_offense.reconcile`:

> Geprüft wird, was stillschweigend schieflaufen kann. Was nur unvollständig
> ist, wird gezählt und nicht beanstandet.

Warum das keine Bequemlichkeit ist: Eine harte Kontrolle auf fehlende
Koordinaten hiesse, dass Chicagos Jahrgang gar nicht erst entsteht, weil 1'748
Delikte keine Koordinate tragen. Eine weiche Kontrolle macht die Lücke
**abfragbar** statt unsichtbar. Hart wird es erst dort, wo die Lücke so gross
wird, dass die Zahl nichts mehr taugt: beim Publish-Gate in Gold.

---

# Stufe 1 · Ingestion

Hier entscheidet sich, ob überhaupt vollständige Daten ankommen. Die Kontrollen
dieser Stufe sind die einzigen, die etwas über die **Quelle** aussagen können;
alles danach sieht nur noch, was angekommen ist.

## 1.1 Der Vollständigkeitsnachweis

**Das Problem.** Eine API mit Pagination kann eine Seite verlieren, ohne dass
etwas nach einem Fehler aussieht. Man bekommt 250'000 Zeilen statt 259'272 und
merkt es nie, weil niemand die richtige Zahl kennt.

**Die Lösung.** Vor dem Download wird die Menge der `id` festgeschrieben.
Geladen wird per Keyset-Pagination. Danach muss die geladene Menge **exakt**
der festgeschriebenen entsprechen: keine fehlt, keine ist zu viel.

In [ ]:
from chicago_pipeline.download_chicago import IngestionError, verify_snapshot

festgeschrieben = [101, 102, 103, 104, 105]

# Der gute Fall: alles angekommen.
geladen = [{"id": i} for i in festgeschrieben]
verify_snapshot(geladen, festgeschrieben)
print(f"vollstaendig: {len(geladen)} von {len(festgeschrieben)} -> kein Fehler")

# Die Gegenprobe: eine Seite ging verloren.
luecke = [{"id": i} for i in festgeschrieben if i != 103]
try:
    verify_snapshot(luecke, festgeschrieben)
except IngestionError as fehler:
    print(f"unvollstaendig -> {type(fehler).__name__}: {fehler}")

Die Fehlermeldung nennt beide Richtungen: *fehlend* und *unerwartet*. Das
zweite ist nicht theoretisch. Liefert die Quelle während des Laufs neue Zeilen
nach, tauchen ids auf, die vorher nicht festgeschrieben waren, und der Lauf
soll auch das melden statt still mehr Daten zu haben als angekündigt.

Im Manifest landet der Nachweis als Zahlenpaar und als Prüfsumme über die
id-Menge:

In [ ]:
ids = sorted(festgeschrieben)
abdruck = hashlib.sha256(",".join(str(i) for i in ids).encode("utf-8")).hexdigest()
print(json.dumps({
    "verified": True,
    "source_count_before": len(ids),
    "source_count_after": len(ids),
    "downloaded_rows": len(ids),
    "snapshot_sha256": abdruck[:16] + "...",
}, indent=2))

**Warum zweimal gezaehlt wird.** `source_count_before` und
`source_count_after` sind zwei Abfragen an die Quelle, vor und nach dem
Download. Unterscheiden sie sich, hat sich die Quelle unter uns bewegt. Der
Lauf scheitert daran nicht, aber er schreibt es ins Manifest, und
`source_changed_during_run` steht dann auf `true`.

## 1.2 Das gepinnte Schema und die Spalte `_extra`

**Das Problem.** Die Stadt kann jederzeit ein Feld hinzufügen oder den Typ
eines Werts ändern. Ein Bronze-Schema, das sich mitbewegt, macht alte und neue
Läufe unvergleichbar. Eines, das stur bleibt, verliert Daten.

**Die Lösung.** Das Schema ist festgelegt. Was nicht hineinpasst, wird nicht
verworfen, sondern wandert unverändert in die Spalte `_extra`.

In [ ]:
from chicago_pipeline.download_chicago import build_schema, fits

schema = build_schema()
print(f"gepinnte Spalten: {len(schema)}")
print("  " + ", ".join(schema.names[:8]) + ", ...")
print()

for feld, wert, was in [
    ("arrest", True, "der erwartete Typ"),
    ("arrest", "ja", "die Stadt schickt plötzlich Text"),
    ("iucr", "0110", "eine gewoehnliche Zeichenkette"),
]:
    passt = fits(feld, wert)
    ziel = "Spalte " + feld if passt else "_extra"
    print(f"  {feld:8} = {wert!r:8} ({was:38}) -> {ziel}")

Nichts geht verloren, und nichts verbiegt das Schema. Der zweite Fall ist
genau der, der ohne `_extra` entweder den Lauf abbrechen oder den Wert
stillschweigend verschlucken würde.

Dazu kommt ein **Abdruck über die Feldtypen der Quelle**. Er steht in jedem
Manifest, und wenn er sich ändert, hat die Stadt etwas am Datensatz getan:

In [ ]:
felder_alt = {"id": "number", "case_number": "text", "date": "calendar_date"}
felder_neu = {"id": "number", "case_number": "text", "date": "text"}

def abdruck_von(felder):
    roh = json.dumps(felder, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(roh.encode("utf-8")).hexdigest()[:16]

print(f"vorher:  {abdruck_von(felder_alt)}")
print(f"nachher: {abdruck_von(felder_neu)}   <- 'date' ist keine Datumsspalte mehr")
print(f"gleich?  {abdruck_von(felder_alt) == abdruck_von(felder_neu)}")

---

# Stufe 2 · Die Ablage

Zwischen Ingestion und Silver liegt das Publizieren. Die Kontrollen hier sagen
nichts über den Inhalt der Daten, sondern über ihre **Unversehrtheit** und
darüber, dass niemand einen Stand überschreibt, den er nicht überschreiben
darf.

Alle folgenden Zellen benutzen `FakeS3Client`, ein S3 im Arbeitsspeicher. Kein
Netz, kein SeaweedFS.

## 2.1 Unveränderliche Läufe

Ein Bronze- oder Raw-Lauf ist ein Beweisstück. Liegt er einmal oben, darf er
sich nicht ändern. Der Versuch, ihn mit anderem Inhalt zu überschreiben, ist
ein Fehler und keine Aktualisierung.

In [ ]:
from storage import Config, FakeS3Client, PublishError, Storage

def ablage(wurzel, backend="s3", klient=None):
    config = Config(backend=backend, local_root=wurzel, s3_endpoint="beispiel",
                    s3_bucket="eimer", s3_access_key="k", s3_secret_key="s")
    return Storage(config, klient or FakeS3Client())

arbeitsordner = tempfile.TemporaryDirectory()
WURZEL = Path(arbeitsordner.name)

lauf = WURZEL / "bronze" / "city=chicago" / "ingest_date=2026-09-15" / "run_A"
lauf.mkdir(parents=True)
(lauf / "part-0001.parquet").write_bytes(b"BEWEISSTUECK")
(lauf / "manifest.json").write_text('{"status": "complete", "run_id": "run_A"}')

s = ablage(WURZEL)
print("erstes Publizieren:", s.publish_run(lauf))

# Derselbe Lauf, anderer Inhalt.
lauf.mkdir(parents=True, exist_ok=True)
(lauf / "part-0001.parquet").write_bytes(b"ETWAS ANDERES")
(lauf / "manifest.json").write_text('{"status": "complete", "run_id": "run_A"}')
try:
    s.publish_run(lauf)
except PublishError as fehler:
    print(f"\nzweites Publizieren -> {type(fehler).__name__}")
    print("  " + str(fehler).replace(". ", ".\n  "))

**Die Reihenfolge ist Teil der Kontrolle.** Innerhalb eines Laufs wird
`manifest.json` **zuletzt** hochgeladen. Ein abgebrochener Upload hinterlässt
damit einen Lauf ohne Manifest, und ein Lauf ohne Manifest gilt nicht. Ein
Leser sieht ihn nicht.

In [ ]:
klient = FakeS3Client()
s2 = ablage(WURZEL, klient=klient)
neu = WURZEL / "bronze" / "city=chicago" / "ingest_date=2026-09-15" / "run_B"
neu.mkdir(parents=True)
for name in ("part-0003.parquet", "part-0001.parquet", "part-0002.parquet"):
    (neu / name).write_bytes(b"x")
(neu / "manifest.json").write_text('{"status": "complete", "run_id": "run_B"}')

for schluessel in s2.publish_run(neu):
    print(" ", schluessel.rsplit("/", 1)[-1])

## 2.2 Gleiche Grösse ist kein gleicher Inhalt

Vor jedem Upload wird geprüft, ob dieselbe Datei schon oben liegt. Das spart
Übertragung, und es ist die Stelle, an der ein Unterschied auffallen muss.

Der Vergleich nimmt die Byteanzahl **und** die Prüfsumme. Das ETag eines
S3-Objekts ist normalerweise dessen MD5 — aber nicht immer: Nach einem
mehrteiligen Upload ist es ein Hash über die Teil-Hashes, mit angehängter
Teilezahl. Dann sagt es über den Inhalt nichts aus, und die Ablage holt ihn
zum Vergleich.

In [ ]:
from storage.backend import Objekt, etag_md5

for etag in ['"d41d8cd98f00b204e9800998ecf8427e"', '"d41d8cd98f00b204e9800998ecf8427e-2"']:
    lesbar = etag_md5(etag)
    print(f"  {etag:40} -> {'MD5: ' + lesbar[:12] + '...' if lesbar else 'keine Pruefsumme'}")

In [ ]:
class EtagOhneMD5(FakeS3Client):
    """Verhaelt sich wie S3 nach einem mehrteiligen Upload."""
    def head(self, key):
        objekt = super().head(key)
        return None if objekt is None else objekt._replace(etag='"abc123-2"')

tabelle = WURZEL / "gold" / "dataset=fact_offense"
# Im s3-Modus entfernt das Publizieren die lokale Arbeitskopie. Fuer die
# naechste Vorfuehrung wird sie deshalb jedes Mal neu angelegt.
tabelle.mkdir(parents=True, exist_ok=True)
(tabelle / "part-0001.parquet").write_bytes(b"NEU")
(tabelle / "manifest.json").write_text('{"status": "complete"}')

k = EtagOhneMD5()
k.objekte["gold/dataset=fact_offense/part-0001.parquet"] = b"ALT"   # gleich lang!
k.objekte["gold/dataset=fact_offense/manifest.json"] = b'{"status": "complete"}'

s3 = ablage(WURZEL, klient=k)
s3.publish_table(tabelle, erzwingen=True)
print("lokal b'NEU' (3 Bytes) gegen entfernt b'ALT' (3 Bytes), ETag ohne MD5")
print("entfernt danach:", k.objekte["gold/dataset=fact_offense/part-0001.parquet"])

Ohne den Inhaltsvergleich hätte die gleiche Byteanzahl als „identisch"
gegolten, der Upload wäre unterblieben, und oben stünde weiterhin `ALT` unter
einem Manifest, das `NEU` behauptet. Eine externe Prüfung hat genau das am
15.9.2026 vorgeführt.

## 2.3 Der Statuswechsel beim Aktualisieren

`manifest.json` zuletzt schützt einen **neuen** Lauf. Eine **bestehende**
Tabelle schützt es nicht: Dort fällt zuerst die Parquet-Datei, und bricht
danach der Manifest-Upload ab, liegen neue Daten unter einem alten Manifest,
das weiterhin `complete` sagt.

Deshalb fällt zuerst der Status. Ein Leser sieht der Reihe nach den
vollständigen alten Stand, dann **keinen**, dann den vollständigen neuen.

In [ ]:
class ManifestFaelltAus(FakeS3Client):
    def put(self, key, daten):
        if key.endswith("manifest.json") and b'"publishing"' not in daten:
            raise OSError("Verbindung abgebrochen")
        super().put(key, daten)

tabelle.mkdir(parents=True, exist_ok=True)
(tabelle / "part-0001.parquet").write_bytes(b"NEUER STAND")
(tabelle / "manifest.json").write_text(
    '{"status": "complete", "built_at_utc": "2026-09-15T10:00:00Z"}')

k2 = ManifestFaelltAus()
k2.objekte["gold/dataset=fact_offense/part-0001.parquet"] = b"ALTER STAND"
k2.objekte["gold/dataset=fact_offense/manifest.json"] = json.dumps(
    {"status": "complete", "built_at_utc": "2026-08-01T10:00:00Z"}).encode()

s4 = ablage(WURZEL, klient=k2)
try:
    s4.publish_table(tabelle, erzwingen=True)
except PublishError as fehler:
    print(f"{type(fehler).__name__}: {str(fehler)[:70]}...")

oben = json.loads(k2.objekte["gold/dataset=fact_offense/manifest.json"])
print(f"\nManifest oben:  status={oben['status']!r}")
print(f"gilt als gebaut: {bool(s4.tabellen(('gold',)))}")

**Was das nicht löst.** Gold besteht aus fünf Tabellen, und sie werden
nacheinander veröffentlicht. Zwischen der ersten und der letzten sieht eine
Abfrage über zwei Tabellen einen gemischten Stand. Dafür bräuchte es
Versionspfade und einen Verweis, der am Ende umgelegt wird; das steht in
`docs/ausblick.md` Abschnitt 6 mit allem, was daran hängt.

## 2.4 Der Standvergleich: kein Rückschritt aus Versehen

Eine abgeleitete Tabelle darf überschrieben werden — aber nicht mit einem
**älteren** Stand. Vor dem Publizieren vergleicht die Ablage den eigenen
Bauzeitpunkt und die Bronze-Läufe, auf denen beide Stände ruhen.

In [ ]:
k3 = FakeS3Client()
k3.objekte["gold/dataset=fact_offense/manifest.json"] = json.dumps({
    "status": "complete",
    "built_at_utc": "2026-09-15T17:00:00Z",
    "derived_from": [{"chicago_bronze_runs": ["run_A", "run_B", "run_C"]}],
}).encode()

tabelle.mkdir(parents=True, exist_ok=True)
(tabelle / "part-0001.parquet").write_bytes(b"IRGENDEIN STAND")
(tabelle / "manifest.json").write_text(json.dumps({
    "status": "complete",
    "built_at_utc": "2026-08-01T09:00:00Z",           # aelter
    "derived_from": [{"chicago_bronze_runs": ["run_A"]}],   # weniger Eingaben
}))

s5 = ablage(WURZEL, klient=k3)
try:
    s5.publish_table(tabelle)
except PublishError as fehler:
    for zeile in str(fehler).split(". "):
        print(" ", zeile.strip())

---

# Stufe 3 · Silver

Silver macht aus zwei Städten eine Tabelle. Hier entstehen die weichen
Kontrollen als Spalten, und hier stehen die harten, die einen stillen
Datenverlust verhindern sollen.

## 3.1 Die weichen Kontrollen: `mapping_status` und `geo_status`

Zwei Fragen lassen sich nicht immer beantworten: *Welches NIBRS-Delikt ist
das?* und *In welchem Tract liegt es?* Statt die Zeile zu verwerfen oder die
Unsicherheit zu verschweigen, steht die Antwort je Zeile in einer Spalte.

In [ ]:
from gold_pipeline import GEO_STATUS_WERTE, MAPPING_STATUS_WERTE

print("mapping_status - wie sicher ist die NIBRS-Zuordnung?")
for wert, bedeutung in [
    ("exact", "ein amtliches Dokument belegt die Zuordnung"),
    ("ambiguous", "mehrere waeren vertretbar, wir haben entschieden"),
    ("fallback_90Z", "Auffangcode, weil keine Kategorie passt"),
    ("unmapped", "keine Zuordnung moeglich"),
]:
    print(f"  {wert:14} {bedeutung}")
print(f"  erlaubte Werte laut Code: {sorted(MAPPING_STATUS_WERTE)}")
print()
print("geo_status - wo liegt das Delikt?")
for wert, bedeutung in [
    ("inside_city", "in einem Tract der meldenden Stadt"),
    ("outside_city", "in einem Tract, der nicht zur Stadt gehoert"),
    ("missing_coordinate", "die Stadt hat keine Koordinate geliefert"),
    ("no_tract", "Koordinate in keinem Tract"),
]:
    print(f"  {wert:19} {bedeutung}")
print(f"  erlaubte Werte laut Code: {sorted(GEO_STATUS_WERTE)}")

**Die Eigenschaft, die das brauchbar macht:** Die Werte summieren sich
auf die Gesamtzahl. Was fehlt, ist gezählt und benannt, nicht weggelassen.
Genau das prüft Gold später hart nach.

Und: Ein Wert, der nicht in dieser Liste steht, lässt den Bau scheitern. Eine
neue Einstufung muss eingetragen werden, sonst ist sie ein Tippfehler.

## 3.2 Die harten Kontrollen: `reconcile`

Geprüft wird, was **stillschweigend** schieflaufen kann. Drei Beispiele, alle
mit einer kleinen synthetischen Tabelle statt mit dem echten Jahrgang.

In [ ]:
import silver_pipeline.build_crime_offense as bc
from silver_pipeline import TransformError

def silbertabelle(zeilen):
    """Eine Minimaltabelle mit genau den Spalten, die reconcile liest."""
    con = duckdb.connect()
    teile = " UNION ALL ".join(
        "SELECT " + ", ".join([
            f"'{z['city']}' AS city", f"'{z['offense_id']}' AS offense_id",
            f"'{z['event_id']}' AS event_id",
            (f"TIMESTAMP '{z['t']}'" if z["t"] else "CAST(NULL AS TIMESTAMP)")
            + " AS occurred_at_utc",
            f"{str(z['counts']).lower()} AS counts",
            f"{str(z['counts_case']).lower()} AS counts_case",
            "true AS nibrs_mapped", "true AS located",
            # Seit Silver-Vertrag 2: die beiden Zeitzonen-Randfaelle.
            "false AS local_time_nonexistent", "false AS local_time_ambiguous",
            f"'{z.get('nibrs_code', '09A')}' AS nibrs_code",
            f"'{z.get('nibrs_match', 'nibrs_definition')}' AS nibrs_match",
        ]) for z in zeilen
    )
    con.execute(f"CREATE TABLE silber AS {teile}")
    return con

def quelle(stadt, zeilen):
    return bc.Source(city=stadt, paths=[], run_ids=["run_A"],
                     row_count=zeilen, reader="", format="PARQUET")

In [ ]:
# Der gute Fall.
con = silbertabelle([
    {"city": "chicago", "offense_id": 1, "event_id": "JH100028",
     "t": "2024-01-01 06:56:00", "counts": True, "counts_case": True},
    {"city": "chicago", "offense_id": 2, "event_id": "JH100028",
     "t": "2024-01-04 02:59:00", "counts": True, "counts_case": False},
    {"city": "detroit", "offense_id": 3, "event_id": "D-1",
     "t": "2024-03-05 12:00:00", "counts": True, "counts_case": True},
])
nachweis = bc.reconcile(con, "silber", quelle("chicago", 2), quelle("detroit", 1))
print("reconcile bestanden. Was er dabei gezaehlt hat:")
for stadt, zahlen in nachweis["per_city"].items():
    print(f"  {stadt:9} {zahlen['rows']} Zeilen, "
          f"{zahlen['distinct_event_id']} Faelle, "
          f"{zahlen['counts']} Taten, {zahlen['counts_case']} zaehlende Faelle")
print(f"  Zeitraum: {nachweis['occurred_at_utc_min']} bis {nachweis['occurred_at_utc_max']}")

### Gegenprobe 1 — ein doppelter Deliktschlüssel

Zwei Zeilen mit derselben `offense_id` wären ein doppelt gezähltes Delikt. Das
sieht man einer Summe nicht an.

In [ ]:
con = silbertabelle([
    {"city": "chicago", "offense_id": 7, "event_id": "A", "t": "2024-01-01 06:00:00",
     "counts": True, "counts_case": True},
    {"city": "chicago", "offense_id": 7, "event_id": "B", "t": "2024-01-02 06:00:00",
     "counts": True, "counts_case": True},
    {"city": "detroit", "offense_id": 9, "event_id": "D-1", "t": "2024-03-05 12:00:00",
     "counts": True, "counts_case": True},
])
try:
    bc.reconcile(con, "silber", quelle("chicago", 2), quelle("detroit", 1))
except TransformError as fehler:
    print(f"{type(fehler).__name__}: {fehler}")

### Gegenprobe 2 — eine Zeile ohne Zeitpunkt

Ohne Zeitpunkt fällt die Zeile aus jeder Zeitreihe heraus, ohne dass jemand
eine Lücke sieht.

In [ ]:
con = silbertabelle([
    {"city": "chicago", "offense_id": 1, "event_id": "A", "t": None,
     "counts": True, "counts_case": True},
    {"city": "detroit", "offense_id": 9, "event_id": "D-1", "t": "2024-03-05 12:00:00",
     "counts": True, "counts_case": True},
])
try:
    bc.reconcile(con, "silber", quelle("chicago", 1), quelle("detroit", 1))
except TransformError as fehler:
    print(f"{type(fehler).__name__}: {fehler}")

### Gegenprobe 3 — mehr Zeilen, als die Quelle hatte

Der Vergleich gegen die Zeilenzahl aus dem Bronze-Manifest fängt einen
fehlerhaften Join ab, der Zeilen vervielfacht.

In [ ]:
con = silbertabelle([
    {"city": "chicago", "offense_id": i, "event_id": f"A{i}",
     "t": "2024-01-01 06:00:00", "counts": True, "counts_case": True}
    for i in range(1, 6)
] + [
    {"city": "detroit", "offense_id": 9, "event_id": "D-1",
     "t": "2024-03-05 12:00:00", "counts": True, "counts_case": True},
])
try:
    bc.reconcile(con, "silber", quelle("chicago", 3), quelle("detroit", 1))
except TransformError as fehler:
    print(f"{type(fehler).__name__}: {fehler}")

## 3.3 Die atomare Ersetzung

Silver wird ersetzt, nicht angehäuft. Trotzdem darf nie ein halber Stand
sichtbar sein. Deshalb wird in ein eigenes Arbeitsverzeichnis geschrieben und
erst am Schluss umbenannt; der Name trägt die Prozessnummer, damit zwei
gleichzeitige Läufe einander nichts wegräumen.

In [ ]:
from silver_pipeline import publish

con = duckdb.connect()
con.execute("CREATE TABLE t AS SELECT 1 AS a")
ziel = WURZEL / "silver" / "dataset=beispiel"
erste = publish(con, "t", ziel, {"status": "complete", "row_count": 1})
print("erster Stand:", sorted(p.name for p in erste.iterdir()))

class BrichtAb:
    def __init__(self, echt): self._echt = echt
    def execute(self, *_, **__): raise RuntimeError("Abbruch mitten im Schreiben")
    def __getattr__(self, name): return getattr(self._echt, name)

try:
    publish(BrichtAb(con), "t", ziel, {"status": "complete", "row_count": 999})
except RuntimeError as fehler:
    print(f"zweiter Versuch: {fehler}")

haltbar = json.loads((ziel / "manifest.json").read_text())
print(f"Stand danach: row_count={haltbar['row_count']} -- der alte steht unveraendert")
print("kein Rest liegengeblieben:", not list(ziel.parent.glob(".*")))

---

# Stufe 4 · Gold

Gold ist die Schicht, auf der ausgewertet wird. Hier steht die einzige
Kontrolle des Projekts, die eine **Veröffentlichung verhindert**, statt nur
einen Fehler zu melden: das Publish-Gate.

## 4.1 Unbekannte Statuswerte

Bevor Anteile gerechnet werden, prüft `abdeckung`, dass jeder vorkommende
Status auch einer der erlaubten ist. Ein neuer Wert ist entweder eine
Entscheidung, die eingetragen gehört, oder ein Tippfehler.

In [ ]:
import gold_pipeline.build_gold as bg

def goldtabelle(zeilen):
    """fact_offense mit den drei Spalten, die Abdeckung und Gate lesen."""
    con = duckdb.connect()
    teile = " UNION ALL ".join(
        f"SELECT '{stadt}' AS reporting_city, '{mapping}' AS mapping_status, "
        f"'{geo}' AS geo_status FROM range({n})"
        for stadt, mapping, geo, n in zeilen
    )
    con.execute(f"CREATE TABLE fact_offense AS {teile}")
    return con

con = goldtabelle([("chicago", "exact", "inside_city", 10),
                   ("chicago", "vielleicht", "inside_city", 1)])
try:
    bg.abdeckung(con)
except TransformError as fehler:
    print(f"{type(fehler).__name__}: {str(fehler)[:150]}...")

## 4.2 Die Anteile messen

`abdeckung` misst nur. Ob die Anteile reichen, entscheidet `pruefe_gate`. Diese
Trennung ist Absicht: Ein Nachweis misst, ein Gate entscheidet.

In [ ]:
con = goldtabelle([
    ("chicago", "exact", "inside_city", 700),
    ("chicago", "fallback_90Z", "outside_city", 50),
    ("detroit", "exact", "inside_city", 240),
    ("detroit", "fallback_90Z", "inside_city", 10),
])
anteile = bg.abdeckung(con)
je_stadt = bg.abdeckung_je_stadt(con)

print(f"{'':10} {'inside_city':>12} {'exact':>9} {'unmapped':>9}")
print(f"{'insgesamt':10} {anteile['inside_city']:>12.2f} {anteile['exact']:>9.2f} "
      f"{anteile['unmapped']:>9.2f}")
for stadt, ihre in je_stadt.items():
    print(f"{stadt:10} {ihre['inside_city']:>12.2f} {ihre['exact']:>9.2f} "
          f"{ihre['unmapped']:>9.2f}")

## 4.3 Das Publish-Gate

Drei Schwellen, und sie liegen **weit** unter dem gemessenen Stand. Sie sollen
einen Einsturz abfangen, nicht die normale Bewegung der Quelle. Eng gezogene
Schwellen wären schlimmer als keine: Sie würden bei jeder Datenbewegung rot,
und die erste Reaktion wäre, sie hochzusetzen statt hinzusehen.

In [ ]:
from gold_pipeline import GATE_HOECHSTENS, GATE_MINDESTENS

print("mindestens:", GATE_MINDESTENS)
print("hoechstens:", GATE_HOECHSTENS)
bg.pruefe_gate(anteile, je_stadt)
print("\n-> beide Staedte halten die Schwellen, Gold darf entstehen")

### Gegenprobe — der räumliche Join bricht

Wenn der Punkt-in-Polygon-Join ins Leere läuft, sind plötzlich fast alle Fälle
`no_tract`. Genau dafür gibt es die Untergrenze bei `inside_city`.

In [ ]:
con = goldtabelle([("chicago", "exact", "no_tract", 900),
                   ("chicago", "exact", "inside_city", 100)])
try:
    bg.pruefe_gate(bg.abdeckung(con), bg.abdeckung_je_stadt(con))
except TransformError as fehler:
    print(f"{type(fehler).__name__}: {str(fehler).split(' Die Schwelle')[0]}")

### Gegenprobe — die grosse Stadt verdeckt die kleine

Das ist der Fall, den eine externe Prüfung am 15.9.2026 gefunden hat. Chicago
stellt drei Viertel aller Zeilen. Ein Gesamtanteil trägt deshalb Chicagos
Zahl, und Detroits Zuordnung könnte zu einem Drittel brechen, ohne dass etwas
rot wird.

In [ ]:
con = goldtabelle([
    ("chicago", "exact", "inside_city", 750),
    ("detroit", "exact", "inside_city", 175),
    ("detroit", "fallback_90Z", "inside_city", 75),
])
anteile = bg.abdeckung(con)
je_stadt = bg.abdeckung_je_stadt(con)
print(f"exact insgesamt: {anteile['exact']} %   Schwelle: {GATE_MINDESTENS['exact']} %")
for stadt, ihre in je_stadt.items():
    print(f"  {stadt:9} {ihre['exact']:>6.1f} %")

print("\nnur der Gesamtanteil:")
bg.pruefe_gate(anteile)
print("  bestanden -- so war es bis zum 15.9.2026")

print("\nmit den Staedtezahlen:")
try:
    bg.pruefe_gate(anteile, je_stadt)
except TransformError as fehler:
    print(f"  {str(fehler).split(' Die Schwelle')[0]}")

---

# Stufe 5 · Serving

Die Auswertungsdatenbank hält keine Daten, sie legt Sichten über die
Parquet-Dateien. Eine Sicht auf eine Datei, die es nicht mehr gibt, fällt sonst
erst bei der ersten Abfrage auf, und dann sitzt jemand davor.

In [ ]:
from serving import build_analysis_db as sv

datenwurzel = WURZEL / "beispielschichten"
tabelle_x = datenwurzel / "gold" / "dataset=fact_offense"
tabelle_x.mkdir(parents=True)
duckdb.connect().execute(
    f"COPY (SELECT 1 AS offense_id, 'chicago' AS reporting_city) "
    f"TO '{tabelle_x}/part-0001.parquet' (FORMAT PARQUET)")

datenbank = sv.build(datenwurzel)
con = duckdb.connect(str(datenbank))
print("Sichten:", [r[0] for r in con.execute(
    "SELECT view_name FROM duckdb_views() WHERE NOT internal ORDER BY 1").fetchall()])
print("fact_offense:", con.execute("SELECT count(*) FROM fact_offense").fetchone()[0], "Zeilen")
con.close()

### Gegenprobe — eine Datei, die keine Parquet-Datei ist

`verify()` liest jede Sicht einmal, bevor die Datenbank an ihren Platz gerückt
wird. Eine halbe Datenbank ist schlimmer als keine: In ihr fehlt ein Teil der
Sichten, ohne dass man es ihr ansieht. Hier bekommt eine zweite Tabelle eine
unlesbare Datei.

In [ ]:
kaputt = datenwurzel / "silver" / "dataset=crime_offense"
kaputt.mkdir(parents=True)
(kaputt / "part-0001.parquet").write_bytes(b"das ist kein Parquet")

try:
    sv.build(datenwurzel)
except RuntimeError as fehler:
    print(f"{type(fehler).__name__}: {str(fehler)[:120]}...")

print("\nalte Datenbank steht unveraendert:", datenbank.is_file())
print("keine halbe Datenbank liegengeblieben:",
      not list(datenbank.parent.glob('.*.part')))
con = duckdb.connect(str(datenbank), read_only=True)
print("und sie ist weiterhin abfragbar:",
      con.execute("SELECT count(*) FROM fact_offense").fetchone()[0], "Zeilen")
con.close()

---

# Stufe 6 · Die Orchestrierung

Die letzten Kontrollen sagen nichts über einzelne Zeilen. Sie sorgen dafür,
dass die Schichten in der richtigen Reihenfolge und auf frischen Daten
entstehen.

## 6.1 Die Reihenfolge ist die Kontrolle

Die Abhängigkeiten sind real: `silver/tract` braucht beide Census-Quellen,
`crime_offense` braucht `tract`, Gold braucht `crime_offense`. Eine falsche
Reihenfolge scheitert nicht, sie liefert eine Rate ohne Nenner.

In [ ]:
from orchestration import run_all as ra

print(f"{len(ra.STUFEN)} Stufen in {len(ra.GRUPPEN)} Gruppen:\n")
letzte = None
for stufe in ra.STUFEN:
    if stufe.gruppe != letzte:
        print(f"  [{stufe.gruppe}]")
        letzte = stufe.gruppe
    fuellt = f"  fuellt {stufe.quelle}" if stufe.quelle else ""
    print(f"    {stufe.name:16} {stufe.modul:38}{fuellt}")

## 6.2 Beim ersten Fehler ist Schluss

Weiterzulaufen wäre schlimmer als abzubrechen: Die nächste Stufe baute auf
einem halben Stand und meldete Erfolg.

In [ ]:
# Die Stufen laufen als Unterprozess mit diesem Ordner als Arbeitsverzeichnis;
# 'python -m eins' findet das Modul dort.
spielwiese = WURZEL / "stufen"
spielwiese.mkdir(exist_ok=True)
spur = spielwiese / "spur.txt"
spur.unlink(missing_ok=True)

for name, code in [("eins", 0), ("zwei", 3), ("drei", 0)]:
    (spielwiese / f"{name}.py").write_text(
        f"import sys\nopen({str(spur)!r}, 'a').write({name!r} + chr(10))\n"
        f"sys.exit({code})\n", encoding="utf-8")

stufen = [ra.Stufe("silver", n, n) for n in ("eins", "zwei", "drei")]
protokoll = ra.fahre(stufen, spielwiese)

print("gelaufen:", spur.read_text().split())
for eintrag in protokoll:
    print(f"  {eintrag['name']:6} exit={eintrag['exit']}")
print("-> 'drei' wurde nie gestartet")

## 6.3 Die Frischeprüfung

Die Quellen bewegen sich. Eine Zahl, die aus drei Wochen altem Bronze
entsteht, sieht genauso aus wie eine frische. Der wöchentliche Datenlauf prüft
deshalb gegen eine Schwelle von acht Tagen.

Zwei Dinge sind daran nicht offensichtlich.

In [ ]:
from datetime import datetime, timedelta, timezone

def bronze_lauf(wurzel, bestand, tage_alt):
    wann = datetime.now(timezone.utc) - timedelta(days=tage_alt)
    ordner = wurzel / "bronze" / bestand / "ingest_date=2026-09-15" / f"run_{bestand}"
    ordner.mkdir(parents=True, exist_ok=True)
    (ordner / "manifest.json").write_text(json.dumps({
        "status": "complete",
        "completed_at_utc": wann.isoformat().replace("+00:00", "Z"),
    }))

frisch = WURZEL / "frischetest"
for bestand in ra.ERWARTETE_QUELLEN:
    bronze_lauf(frisch, bestand, tage_alt=1)

print("alle sechs Quellen einen Tag alt:")
print(" ", ra.veraltete_quellen(frisch, 8.0) or "nichts zu beanstanden")

**Erstens: Eine Quelle, die ganz fehlt, ist nicht frisch — sie ist nicht
da.** Geprüft wird nicht, was vorhanden ist, sondern was vorhanden sein
*müsste*. Die Liste dafür kommt aus den Stufen selbst.

In [ ]:
import shutil

shutil.rmtree(frisch / "bronze" / "dataset=noaa_ghcnd")
bronze_lauf(frisch, "city=chicago", tage_alt=30)

for bestand, tage in ra.veraltete_quellen(frisch, 8.0).items():
    print(" ", ra.alterssatz(bestand, tage, 8.0))

**Zweitens: Die Prüfung liegt vor dem Ableiten, nicht danach.** Bis zum
15.9.2026 lief sie nach allen dreizehn Stufen. Der Lauf endete mit `failed`,
aber Gold und Serving waren gebaut — aus Daten, die er selbst als zu alt
erkannt hatte. Eine Kontrolle, die nach der Veröffentlichung greift, ist ein
Protokoll und keine Kontrolle.

In [ ]:
from unittest import mock

befehle = []

class Fertig:
    returncode = 0

with mock.patch.object(ra.subprocess, "run",
                       lambda cmd, cwd=None, **_: (befehle.append(cmd), Fertig())[1]):
    manifest = ra.lauf(frisch, PROJECT_ROOT, maximal_alter_tage=8.0)

gefahren = {c[2] for c in befehle}          # [python, -m, modul, ...]
gruppen = sorted({s.gruppe for s in ra.STUFEN if s.modul in gefahren})
print(f"gefahren: {len(befehle)} von {len(ra.STUFEN)} Stufen")
print(f"  nur diese Gruppen: {gruppen}")
print(f"Endstatus: {manifest['status']}")
print(f"output_rows: {manifest['output_rows']}  <- nichts abgeleitet")

Nebenbei sichtbar: Jede Befehlszeile trägt `--data-root`. Vater und Kind
arbeiten damit garantiert in derselben Wurzel.

In [ ]:
print(" ".join(befehle[0][1:]))

---

# Ein Fall von Anfang bis Ende

`JH100028` ist der erste Mordfall des Jahrgangs 2024 in Chicago. Er eignet
sich, weil er **zwei Zeilen** hat: Chicago führt je Opfer einen Datensatz. An
ihm lässt sich zeigen, welche Kontrolle ihn auf welcher Stufe angesehen hat
und was sie gesagt hat.

Der Fall wird hier synthetisch nachgebaut, mit den Werten, die er im echten
Jahrgang trägt.

In [ ]:
fall = [
    {"id": 27932, "case_number": "JH100028", "date": "2024-01-01T00:56:00",
     "iucr": "0110", "arrest": False,
     "latitude": "41.79", "longitude": "-87.60"},
    {"id": 27933, "case_number": "JH100028", "date": "2024-01-03T20:59:00",
     "iucr": "0110", "arrest": False,
     "latitude": "41.79", "longitude": "-87.60"},
]
for zeile in fall:
    print(f"  id={zeile['id']}  {zeile['case_number']}  {zeile['date']}  IUCR {zeile['iucr']}")

## Stufe 1 — Ingestion

Beide ids waren vorher festgeschrieben, beide sind angekommen. Die Typen passen
zum gepinnten Schema, also bleibt `_extra` leer.

In [ ]:
festgeschrieben = [z["id"] for z in fall]
verify_snapshot(fall, festgeschrieben)
print(f"Vollstaendigkeitsnachweis: {len(fall)} von {len(festgeschrieben)} id(s) -> bestanden")

nach_extra = [(f, z[f]) for z in fall[:1] for f in ("arrest", "iucr", "date")
              if not fits(f, z[f])]
print(f"Werte, die ins _extra muessten: {nach_extra or 'keine'}")

## Stufen 2 bis 3 — Ablage und Silver

In Silver bekommt der Fall drei Spalten, die Kontrollen sind: `mapping_status`
sagt, wie sicher die NIBRS-Zuordnung ist, `counts` und `counts_case` sagen, ob
die Zeile als Tat und als Fall zählt.

Die Hierarchieregel lässt je Fall genau **eine** Zeile als Fall zählen, und
zwar die frühere. Das zweite Opfer starb zwei Tage später.

In [ ]:
silber = [
    {"city": "chicago", "offense_id": "27932", "event_id": "JH100028",
     "t": "2024-01-01 06:56:00", "counts": True, "counts_case": True},
    {"city": "chicago", "offense_id": "27933", "event_id": "JH100028",
     "t": "2024-01-04 02:59:00", "counts": True, "counts_case": False},
    {"city": "detroit", "offense_id": "D-1", "event_id": "D-1",
     "t": "2024-03-05 12:00:00", "counts": True, "counts_case": True},
]
con = silbertabelle(silber)
nachweis = bc.reconcile(con, "silber", quelle("chicago", 2), quelle("detroit", 1))
chi = nachweis["per_city"]["chicago"]
print(f"  Zeilen                 {chi['rows']}")
print(f"  verschiedene offense_id {chi['distinct_offense_id']}  <- Granularitaet haelt")
print(f"  Faelle                 {chi['distinct_event_id']}")
print(f"  zaehlen als Tat        {chi['counts']}")
print(f"  zaehlen als Fall       {chi['counts_case']}  <- genau einer")
print(f"  Zeilen ohne Zeitpunkt  {chi['without_timestamp']}")

Die Ortszeit 00:56 ist UTC 06:56. Beim zweiten Opfer springt die UTC-Zeit
sogar auf den Folgetag. Ohne die Umrechnung lägen die beiden Städte auf
verschiedenen Zeitachsen, und `hour_of_day` wäre falsch.

**Was hier absichtlich nicht scheitert:** Hätte eine der beiden Zeilen keine
Koordinate, stünde `located = false` und `geo_status = missing_coordinate`. Der
Bau liefe weiter. Erst wenn das so vielen Zeilen passiert, dass die Rate nichts
mehr taugt, greift das Gate in Gold.

## Stufe 4 — Gold

In Gold zählt nur, was `counts` trägt: beide Zeilen. `geo_status` entsteht hier
neu aus Koordinate und Tract-Zuordnung.

In [ ]:
con = goldtabelle([("chicago", "exact", "inside_city", 2),
                   ("detroit", "exact", "inside_city", 1)])
anteile = bg.abdeckung(con)
je_stadt = bg.abdeckung_je_stadt(con)
bg.pruefe_gate(anteile, je_stadt)
print(f"  inside_city {anteile['inside_city']:.0f} %, exact {anteile['exact']:.0f} % "
      f"-> Gate bestanden, Gold darf entstehen")

## Was der Fall unterwegs an Kontrollen passiert hat

| Stufe | Kontrolle | Was sie zu diesem Fall sagte | Art |
| --- | --- | --- | --- |
| Ingestion | `verify_snapshot` | beide ids angekommen | hart |
| Ingestion | `fits` / `_extra` | alle Werte passen ins gepinnte Schema | weich |
| Ingestion | Schemaabdruck | die Quelle hat ihre Feldtypen nicht geändert | weich |
| Ablage | `_ist_identisch` | der Lauf lag noch nicht oben | hart |
| Ablage | Manifest zuletzt | der Lauf gilt erst, wenn alles oben ist | hart |
| Silver | `mapping_status` | `exact`, ein amtliches Dokument belegt `0110` → `09A` | weich |
| Silver | `geo_status` | Koordinate vorhanden, Tract gefunden | weich |
| Silver | `counts` / `counts_case` | 2 Taten, 1 Fall | weich |
| Silver | `reconcile` Granularität | 2 Zeilen, 2 verschiedene `offense_id` | hart |
| Silver | `reconcile` Zeitpunkt | keine Zeile ohne Zeitpunkt | hart |
| Silver | `reconcile` Fallzahl | `counts_case` = Zahl gezählter Fälle | hart |
| Gold | Statuswerte | beide Werte sind bekannt | hart |
| Gold | `pruefe_gate` | Abdeckung hält, insgesamt und je Stadt | hart |
| Serving | `verify` | die Sicht ist lesbar und liefert Zeilen | hart |
| Orchestrierung | Frischeprüfung | die Quelle war jünger als acht Tage | hart |

Zwei der Zeilen sind bemerkenswert. `counts_case` ist **weich** und trotzdem
die Spalte, die den ganzen Vergleich verschiebt: Auf Fallebene hat Chicago 29
Delikte weniger als auf Tatebene. Und `reconcile` Fallzahl ist **hart**,
obwohl es dieselbe Spalte prüft — weil ein Fall, der beim Zählen verlorengeht,
sonst niemandem auffiele.

---

# Die Landkarte

Alle Kontrollen der Strecke, nach Schicht geordnet. Die Liste ist keine
Beschreibung, sondern eine Datenstruktur: Die nächste Zelle prüft, dass jeder
genannte Name im Code auch existiert. Wird eine Funktion umbenannt und diese
Liste nicht, fällt das Notebook.

In [ ]:
# (Schicht, Ort im Code, was geprueft wird, hart/weich)
KONTROLLEN = [
    # --- Ingestion, Chicago ------------------------------------------------
    ("Ingestion", "chicago_pipeline.download_chicago.snapshot_ids",
     "Antwort ist eine Liste, jede id ein int, keine Dubletten", "hart"),
    ("Ingestion", "chicago_pipeline.download_chicago.verify_snapshot",
     "geladene id-Menge == vorher festgeschriebene", "hart"),
    ("Ingestion", "chicago_pipeline.download_chicago.expected_rows",
     "serverseitiger count; nicht verfuegbar -> verified=false", "weich"),
    ("Ingestion", "chicago_pipeline.download_chicago.source_schema",
     "SHA-256 ueber die Feldtypen der Quelle", "weich"),
    ("Ingestion", "chicago_pipeline.download_chicago.fits",
     "JSON-Typ passt zum gepinnten Schema, sonst _extra", "weich"),
    ("Ingestion", "chicago_pipeline.download_chicago.discard_stale_partials",
     ".part aelter als 24 h wird entfernt", "weich"),

    # --- Ingestion, Detroit ------------------------------------------------
    ("Ingestion", "detroit_pipeline.detroit_arcgis.ArcGISClient.get_object_ids",
     "keine doppelten Object-IDs", "hart"),
    ("Ingestion", "detroit_pipeline.bronze_manifest.validate_manifest",
     "Pruefsumme, fuenf Zaehlungen, Speicherschema, Zeilen-Lineage", "hart"),
    ("Ingestion", "detroit_pipeline.bronze_parquet.schema_hash",
     "SHA-256 ueber das physische Speicherschema", "weich"),
    ("Ingestion", "detroit_pipeline.load_duckdb.load_run",
     "bekannte run_id mit anderer Pruefsumme -> Abbruch", "hart"),

    # --- Ingestion, Census und Wetter --------------------------------------
    ("Ingestion", "census_pipeline.download_tiger.verify_archive",
     "ZIP-CRC je Datei, Pflichtdateien vollstaendig", "hart"),
    ("Ingestion", "census_pipeline.download_acs.parse_payload",
     "Antwort statt Status pruefen: faengt 'Missing Key' mit HTTP 200", "hart"),
    ("Ingestion", "census_pipeline.download_acs.verify_tracts",
     "keine doppelten GEOIDs", "hart"),
    ("Ingestion", "census_pipeline.download_acs.compare_with_tiger",
     "ACS-GEOIDs gegen TIGER derselben Vintage", "weich"),
    ("Ingestion", "weather_pipeline.download_noaa.read_series",
     "Pflichtspalten, keine fremden Stationen in der Datei", "hart"),

    # --- Ablage ------------------------------------------------------------
    ("Ablage", "storage.backend.Storage._ist_identisch",
     "Groesse, dann Pruefsumme, sonst Inhaltsvergleich", "hart"),
    ("Ablage", "storage.backend.Storage.publish_files",
     "bestehendes Objekt mit anderem Inhalt -> Abbruch; Manifest zuletzt", "hart"),
    ("Ablage", "storage.backend.Storage._entwerte_entferntes_manifest",
     "Status faellt auf publishing, bevor Daten ueberschrieben werden", "hart"),
    ("Ablage", "storage.backend.Storage.publish_table",
     "kein Rueckschritt: aelterer Stand oder weniger Bronze-Laeufe", "hart"),
    ("Ablage", "storage.backend.Storage.tabellen",
     "nur Manifeste mit status complete gelten als gebaut", "weich"),
    ("Ablage", "storage.config.load",
     "Backend bekannt, Remote-Betrieb hat Schluessel und Bucket", "hart"),

    # --- Silver ------------------------------------------------------------
    ("Silver", "silver_pipeline.newest_complete_run",
     "nur Laeufe mit fertigem Manifest und vorhandener Parquet-Datei", "hart"),
    ("Silver", "silver_pipeline.publish",
     "atomares Ersetzen ueber .part mit Prozessnummer", "hart"),
    ("Silver", "silver_pipeline.build_tract.reconcile",
     "geoid eindeutig; ACS-Attribute ohne Geometrie", "hart"),
    ("Silver", "silver_pipeline.build_place.reconcile",
     "beide Studienstaedte vorhanden, Name passt zur GEOID", "hart"),
    ("Silver", "silver_pipeline.build_weather_day.reconcile",
     "(city, date) eindeutig; Bronze-Zeilen == Silver-Zeilen", "hart"),
    ("Silver", "silver_pipeline.build_crime_offense.reconcile",
     "Granularitaet, Zeitpunkt, Fallzahl, Herkunft, nicht mehr als die Quelle",
     "hart"),
    ("Silver", "silver_pipeline.build_crime_offense.MAPPING_STATUS",
     "Guete der NIBRS-Zuordnung je Zeile", "weich"),

    # --- Gold --------------------------------------------------------------
    ("Gold", "gold_pipeline.build_gold.abdeckung",
     "jeder Statuswert ist ein erlaubter", "hart"),
    ("Gold", "gold_pipeline.build_gold.abdeckung_je_stadt",
     "dieselben Anteile je meldender Stadt", "weich"),
    ("Gold", "gold_pipeline.build_gold.pruefe_gate",
     "Publish-Gate, insgesamt und je Stadt", "hart"),
    ("Gold", "gold_pipeline.build_gold.reconcile",
     "Silver-counts == Faktenzeilen, keine verwaisten Schluessel", "hart"),
    ("Gold", "gold_pipeline.GATE_MINDESTENS",
     "Untergrenzen inside_city und exact", "hart"),

    # --- Serving -----------------------------------------------------------
    ("Serving", "serving.build_analysis_db.verify",
     "jede Sicht lesbar und liefert Zeilen", "hart"),
    ("Serving", "serving.build_analysis_db.uebernimm_eigene_tabellen",
     "eigene Tabelle mit Sichtnamen wird uebergangen", "weich"),
    ("Serving", "serving.export_geojson.sammle",
     "Geometrie nach ST_Simplify gueltig und nicht leer", "hart"),

    # --- Orchestrierung ----------------------------------------------------
    ("Orchestrierung", "orchestration.run_all.fahre",
     "erste gescheiterte Stufe bricht die Kette ab", "hart"),
    ("Orchestrierung", "orchestration.run_all.veraltete_quellen",
     "Alter je Quelle gegen Schwelle; fehlende Quelle zaehlt als nicht da",
     "hart"),
    ("Orchestrierung", "orchestration.run_all.ERWARTETE_QUELLEN",
     "welche Bestaende dasein muessen, abgeleitet aus den Stufen", "hart"),
    ("Orchestrierung", "orchestration.aufraeumen.gepinnte_laeufe",
     "Referenzstand wird nie aufgeraeumt", "hart"),
    ("Orchestrierung", "orchestration.drift.MAX_RATE",
     "Abweichung vom Referenzstand gegen Toleranz", "weich"),
]
print(f"{len(KONTROLLEN)} Kontrollen erfasst")

In [ ]:
import importlib

fehlend = []
for _, ort, _, _ in KONTROLLEN:
    modulname, _, rest = ort.rpartition(".")
    try:
        obj = importlib.import_module(modulname)
    except ModuleNotFoundError:
        # Klassenmitglied: noch einen Punkt zurueck.
        modulname, _, klasse = modulname.rpartition(".")
        obj = getattr(importlib.import_module(modulname), klasse, None)
    if obj is None or not hasattr(obj, rest):
        fehlend.append(ort)

if fehlend:
    raise AssertionError("im Code nicht gefunden: " + ", ".join(fehlend))
print("Alle genannten Namen existieren im Code.")

In [ ]:
from collections import Counter

schichten = ["Ingestion", "Ablage", "Silver", "Gold", "Serving", "Orchestrierung"]
print(f"{'Schicht':16} {'hart':>5} {'weich':>6} {'gesamt':>7}")
print("-" * 37)
for schicht in schichten:
    zeilen = [k for k in KONTROLLEN if k[0] == schicht]
    art = Counter(k[3] for k in zeilen)
    print(f"{schicht:16} {art['hart']:>5} {art['weich']:>6} {len(zeilen):>7}")
art = Counter(k[3] for k in KONTROLLEN)
print("-" * 37)
print(f"{'':16} {art['hart']:>5} {art['weich']:>6} {len(KONTROLLEN):>7}")

In [ ]:
letzte = None
for schicht, ort, was, wie in KONTROLLEN:
    if schicht != letzte:
        print(f"\n## {schicht}")
        letzte = schicht
    marke = "!" if wie == "hart" else "."
    print(f"  {marke} {ort.split('.')[-1]:32} {was}")

## Was dabei auffällt

**Die Ingestion trägt die meisten harten Kontrollen.** Das ist kein Zufall:
Nur dort lässt sich noch etwas über die **Quelle** sagen. Ab Bronze sieht jede
weitere Schicht nur, was angekommen ist. Wer die Vollständigkeit nicht beim
Laden nachweist, kann sie später nicht mehr nachweisen, sondern nur noch
behaupten.

**Genau eine Kontrolle verhindert eine Veröffentlichung.** Das Publish-Gate in
Gold. Alle anderen harten Kontrollen brechen den Bau ab, bevor überhaupt etwas
entsteht; das Gate entscheidet über etwas Fertiges.

**Die weichen Kontrollen sind keine schwächeren Kontrollen.** Sie beantworten
eine andere Frage. `geo_status` verhindert nichts, aber ohne ihn wüsste
niemand, dass 1'748 Chicagoer Delikte keine Koordinate haben. Die Lücke ist
damit abfragbar statt unsichtbar, und die Entscheidung, ob sie das Ergebnis
trägt, liegt bei dem, der die Abfrage schreibt.

**Eine Asymmetrie bleibt.** Der Mengenabgleich zwischen ACS und TIGER in
`census_pipeline.download_acs.compare_with_tiger` ist der einzige
Vollständigkeitsvergleich, der bei Abweichung nur warnt. `verified` bleibt
dabei `true`. Das ist vertretbar, weil die beiden Quellen verschiedene
Vintages haben können, aber es ist eine Ausnahme von der Regel und gehört
gesagt.

---

## Wo das hier weiterführt

* Wie eine einzelne Zahl durch alle Schichten zurückverfolgt wird:
  [`docs/eine-zahl.md`](../docs/eine-zahl.md)
* Wie die Pakete zusammenhängen: [`docs/architektur.md`](../docs/architektur.md)
* Was jede Spalte bedeutet: [`docs/feldkatalog.md`](../docs/feldkatalog.md)
* Jede Funktion mit ihrem Docstring: [`docs/referenz/`](../docs/referenz/index.html)
* Was sich wann geändert hat und warum:
  [`docs/AENDERUNGEN.md`](../docs/AENDERUNGEN.md)

In [ ]:
arbeitsordner.cleanup()
print("Arbeitsordner entfernt.")